# 타이타닉 데이터를 지식그래프용 CSV로 변환하기

타이타닉 데이터셋(train.csv): https://www.kaggle.com/c/titanic/data

In [1]:
import pandas as pd

df = pd.read_csv('./train.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C




## 그래프 스키마 설계

**노드:**
- Passenger: PassengerId, Name, Sex, Age, Survived, SibSp, Parch, Fare, Ticket
- PClass: Pclass, ClassName, SES (사회경제적 지위)
- Cabin: Cabin  
- Port: Port, PortName

**관계:**
- Passenger → PClass (TRAVELED_IN) - 승객이 어떤 등급으로 여행했는지
- Passenger → Cabin (STAYED_IN) - 승객이 어느 선실에 머물렀는지
- Passenger → Port (EMBARKED_AT) - 승객이 어느 항구에서 승선했는지
- Passenger ↔ Passenger (TRAVELED_WITH) - 같은 티켓을 공유한 그룹/가족

In [2]:
# 데이터 기본 정보 확인
print(f"전체 데이터 개수: {len(df)}")
print(f"\n컬럼 정보:")
df.info()
print(f"\n결측치 확인:")
df.isnull().sum()

전체 데이터 개수: 891

컬럼 정보:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

결측치 확인:


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [3]:
# 같은 티켓 번호에 대해 fare(가격)이 같은지 확인
print("티켓별 요금 일관성 검사")
print("=" * 50)

# 티켓별로 그룹화하여 요금의 고유값 개수 확인
ticket_fare_check = df.groupby('Ticket')['Fare'].agg(['nunique', 'min', 'max', 'count']).reset_index()
ticket_fare_check.columns = ['Ticket', 'unique_fares', 'min_fare', 'max_fare', 'passenger_count']

# 같은 티켓에 서로 다른 요금이 있는 경우
inconsistent_tickets = ticket_fare_check[ticket_fare_check['unique_fares'] > 1]

print(f"전체 고유 티켓 수: {len(ticket_fare_check):,}")
print(f"요금이 일치하지 않는 티켓 수: {len(inconsistent_tickets):,}")
print(f"요금이 일치하는 티켓 수: {len(ticket_fare_check) - len(inconsistent_tickets):,}")

if len(inconsistent_tickets) > 0:
    print(f"\n요금 불일치 티켓 상세:")
    print(inconsistent_tickets.sort_values('unique_fares', ascending=False))

    # 불일치 사례 중 몇 개 상세 확인
    print(f"\n불일치 사례 예시:")
    for idx, ticket_num in enumerate(inconsistent_tickets['Ticket'].head(3)):
        print(f"\n{idx+1}. 티켓 '{ticket_num}':")
        ticket_details = df[df['Ticket'] == ticket_num][['PassengerId', 'Name', 'Ticket', 'Fare']].sort_values('Fare')
        for _, row in ticket_details.iterrows():
            print(f"   승객 {row['PassengerId']}: {row['Name'][:30]:<30} → ${row['Fare']:.2f}")
else:
    print("\n모든 티켓의 요금이 일치합니다!")

티켓별 요금 일관성 검사
전체 고유 티켓 수: 681
요금이 일치하지 않는 티켓 수: 1
요금이 일치하는 티켓 수: 680

요금 불일치 티켓 상세:
    Ticket  unique_fares  min_fare  max_fare  passenger_count
504   7534             2    9.2167    9.8458                2

불일치 사례 예시:

1. 티켓 '7534':
   승객 139: Osen, Mr. Olaf Elon            → $9.22
   승객 877: Gustafsson, Mr. Alfred Ossian  → $9.85


### 결론:
- 같은 티켓 번호라도 요금이 다를 수 있음 (1개 티켓에 2개 다른 요금 발견)
- **Ticket을 노드로 하고 Fare를 속성으로 두는 것은 부적절**
- **대신 PClass(등급)를 노드로 활용하고, Fare는 Passenger 속성으로 관리**
- Ticket은 가족/그룹 여행을 나타내는 **관계(TRAVELED_WITH)**로 활용

In [4]:
# 같은 티켓을 공유하는 그룹 분석
ticket_group_size = df['Ticket'].value_counts()
print("티켓 그룹 분석 (가족/친구 여행)")
print("=" * 50)
print(f"단독 여행 (1명): {(ticket_group_size == 1).sum()}개 티켓")
print(f"그룹 여행 (2명 이상): {(ticket_group_size > 1).sum()}개 티켓")
print(f"\n그룹 크기별 분포:")
for size in sorted(ticket_group_size.unique(), reverse=True)[:5]:
    count = (ticket_group_size == size).sum()
    print(f"  {size}명 그룹: {count}개")

# 가장 큰 그룹 예시
largest_ticket = ticket_group_size.idxmax()
largest_group = df[df['Ticket'] == largest_ticket][['PassengerId', 'Name', 'Fare', 'SibSp', 'Parch']]
print(f"\n가장 큰 그룹 예시 (티켓: {largest_ticket}, {len(largest_group)}명):")
print(largest_group.to_string(index=False))

티켓 그룹 분석 (가족/친구 여행)
단독 여행 (1명): 547개 티켓
그룹 여행 (2명 이상): 134개 티켓

그룹 크기별 분포:
  7명 그룹: 3개
  6명 그룹: 3개
  5명 그룹: 2개
  4명 그룹: 11개
  3명 그룹: 21개

가장 큰 그룹 예시 (티켓: 347082, 7명):
 PassengerId                                                      Name   Fare  SibSp  Parch
          14                               Andersson, Mr. Anders Johan 31.275      1      5
         120                         Andersson, Miss. Ellis Anna Maria 31.275      4      2
         542                      Andersson, Miss. Ingeborg Constanzia 31.275      4      2
         543                         Andersson, Miss. Sigrid Elisabeth 31.275      4      2
         611 Andersson, Mrs. Anders Johan (Alfrida Konstantia Brogren) 31.275      1      5
         814                        Andersson, Miss. Ebba Iris Alfrida 31.275      4      2
         851                   Andersson, Master. Sigvard Harald Elias 31.275      4      2


## 1. 노드 CSV 생성

In [5]:
# 1) Passenger 노드
nodes_passenger = df[[
    'PassengerId', 'Name', 'Sex', 'Age',
    'Survived', 'SibSp', 'Parch', 'Fare', 'Ticket'
]].copy()

# PassengerId를 문자열로 변환 (Neo4j에서 ID로 사용)
nodes_passenger['PassengerId'] = 'P' + nodes_passenger['PassengerId'].astype(str)

print(f"Passenger 노드 개수: {len(nodes_passenger)}")
nodes_passenger.head()

Passenger 노드 개수: 891


,PassengerId,Name,Sex,Age,Survived,SibSp,Parch,Fare,Ticket
0,P1,"Braund, Mr. Owen Harris",male,22.0,0,1,0,7.2500,A/5 21171
1,P2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,1,0,71.2833,PC 17599
2,P3,"Heikkinen, Miss. Laina",female,26.0,1,0,0,7.9250,STON/O2. 3101282
3,P4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,1,0,53.1000,113803
4,P5,"Allen, Mr. William Henry",male,35.0,0,0,0,8.0500,373450


In [6]:
# 2) PClass 노드 - 승객 등급 (사회경제적 지위의 대리 지표)
pclass_mapping = {
    1: {'ClassName': '1st Class', 'SES': 'Upper'},
    2: {'ClassName': '2nd Class', 'SES': 'Middle'},
    3: {'ClassName': '3rd Class', 'SES': 'Lower'}
}

nodes_pclass = pd.DataFrame([
    {'Pclass': k, 'ClassName': v['ClassName'], 'SES': v['SES']}
    for k, v in pclass_mapping.items()
])

print(f"PClass 노드 개수: {len(nodes_pclass)}")
nodes_pclass

PClass 노드 개수: 3


,Pclass,ClassName,SES
0,1,1st Class,Upper
1,2,2nd Class,Middle
2,3,3rd Class,Lower


In [7]:
# 3) Cabin 노드 - 고유한 선실 번호
nodes_cabin = df[['Cabin']].drop_duplicates().copy()
nodes_cabin = nodes_cabin[nodes_cabin['Cabin'].notna()]

print(f"Cabin 노드 개수: {len(nodes_cabin)}")
nodes_cabin.head()

Cabin 노드 개수: 147


,Cabin
1,C85
3,C123
6,E46
10,G6
11,C103


In [8]:
# 4) Port 노드 - 승선 항구 (S=Southampton, C=Cherbourg, Q=Queenstown)
port_mapping = {
    'S': 'Southampton',
    'C': 'Cherbourg',
    'Q': 'Queenstown'
}

nodes_port = df[['Embarked']].drop_duplicates().copy()
nodes_port = nodes_port[nodes_port['Embarked'].notna()]
nodes_port = nodes_port.rename(columns={'Embarked': 'Port'})
nodes_port['PortName'] = nodes_port['Port'].map(port_mapping)

print(f"Port 노드 개수: {len(nodes_port)}")
nodes_port

Port 노드 개수: 3


,Port,PortName
0,S,Southampton
1,C,Cherbourg
5,Q,Queenstown


## 2. 관계 CSV 생성

In [9]:
# 1) Passenger → PClass (TRAVELED_IN)
rels_passenger_pclass = df[['PassengerId', 'Pclass']].copy()
rels_passenger_pclass['PassengerId'] = 'P' + rels_passenger_pclass['PassengerId'].astype(str)

print(f"Passenger-PClass 관계 개수: {len(rels_passenger_pclass)}")
rels_passenger_pclass.head()

Passenger-PClass 관계 개수: 891


,PassengerId,Pclass
0,P1,3
1,P2,1
2,P3,3
3,P4,1
4,P5,3


In [10]:
# 2) Passenger → Cabin (STAYED_IN)
rels_passenger_cabin = df[['PassengerId', 'Cabin']].copy()
rels_passenger_cabin['PassengerId'] = 'P' + rels_passenger_cabin['PassengerId'].astype(str)
rels_passenger_cabin = rels_passenger_cabin[rels_passenger_cabin['Cabin'].notna()]

print(f"Passenger-Cabin 관계 개수: {len(rels_passenger_cabin)}")
rels_passenger_cabin.head()

Passenger-Cabin 관계 개수: 204


,PassengerId,Cabin
1,P2,C85
3,P4,C123
6,P7,E46
10,P11,G6
11,P12,C103


In [11]:
# 3) Passenger → Port (EMBARKED_AT)
rels_passenger_port = df[['PassengerId', 'Embarked']].copy()
rels_passenger_port['PassengerId'] = 'P' + rels_passenger_port['PassengerId'].astype(str)
rels_passenger_port = rels_passenger_port[rels_passenger_port['Embarked'].notna()]
rels_passenger_port = rels_passenger_port.rename(columns={'Embarked': 'Port'})

print(f"Passenger-Port 관계 개수: {len(rels_passenger_port)}")
rels_passenger_port.head()

Passenger-Port 관계 개수: 889


,PassengerId,Port
0,P1,S
1,P2,C
2,P3,S
3,P4,S
4,P5,S


In [12]:
# 4) Passenger ↔ Passenger (TRAVELED_WITH) - 같은 티켓 그룹
# 2명 이상이 같은 티켓을 공유하는 경우 관계 생성
ticket_groups = df.groupby('Ticket')['PassengerId'].apply(list).reset_index() # Ticket 기준으로 그룹화
ticket_groups = ticket_groups[ticket_groups['PassengerId'].apply(len) > 1] # 2명 이상인 그룹만 남김

rels_traveled_with = []
# 각 그룹에 대해 모든 승객 쌍 생성
for _, row in ticket_groups.iterrows():
    passengers = row['PassengerId']
    for i in range(len(passengers)): # 조합(combination)
        for j in range(i+1, len(passengers)):
            rels_traveled_with.append({
                'PassengerId1': f"P{passengers[i]}",
                'PassengerId2': f"P{passengers[j]}",
                'Ticket': row['Ticket']
            })

rels_traveled_with = pd.DataFrame(rels_traveled_with)

print(f"Passenger-Passenger (TRAVELED_WITH) 관계 개수: {len(rels_traveled_with)}")
rels_traveled_with.head()

Passenger-Passenger (TRAVELED_WITH) 관계 개수: 351


,PassengerId1,PassengerId2,Ticket
0,P258,P505,110152
1,P258,P760,110152
2,P505,P760,110152
3,P263,P559,110413
4,P263,P586,110413


## 3. CSV 파일로 저장

In [13]:
import os

output_dir = './output'
os.makedirs(output_dir, exist_ok=True)

# 노드 CSV 저장
nodes_passenger.to_csv(f'{output_dir}/nodes_passenger.csv', index=False, encoding='utf-8-sig')
nodes_pclass.to_csv(f'{output_dir}/nodes_pclass.csv', index=False, encoding='utf-8-sig')
nodes_cabin.to_csv(f'{output_dir}/nodes_cabin.csv', index=False, encoding='utf-8-sig')
nodes_port.to_csv(f'{output_dir}/nodes_port.csv', index=False, encoding='utf-8-sig')

# 관계 CSV 저장
rels_passenger_pclass.to_csv(f'{output_dir}/rels_passenger_pclass.csv', index=False, encoding='utf-8-sig')
rels_passenger_cabin.to_csv(f'{output_dir}/rels_passenger_cabin.csv', index=False, encoding='utf-8-sig')
rels_passenger_port.to_csv(f'{output_dir}/rels_passenger_port.csv', index=False, encoding='utf-8-sig')
rels_traveled_with.to_csv(f'{output_dir}/rels_traveled_with.csv', index=False, encoding='utf-8-sig')

print("CSV 파일 저장 완료!")
print(f"\n저장된 파일 목록:")
for filename in sorted(os.listdir(output_dir)):
    if filename.endswith('.csv'):
        filepath = os.path.join(output_dir, filename)
        line_count = sum(1 for _ in open(filepath, encoding='utf-8-sig')) - 1  # 헤더 제외
        print(f"  - {filename}: {line_count:,} rows")

CSV 파일 저장 완료!

저장된 파일 목록:
  - nodes_cabin.csv: 147 rows
  - nodes_passenger.csv: 891 rows
  - nodes_pclass.csv: 3 rows
  - nodes_port.csv: 3 rows
  - rels_passenger_cabin.csv: 204 rows
  - rels_passenger_pclass.csv: 891 rows
  - rels_passenger_port.csv: 889 rows
  - rels_traveled_with.csv: 351 rows


## 4. 생성된 그래프 구조 요약

In [14]:
print("타이타닉 지식그래프 구조")
print("=" * 60)
print(f"\n[노드 유형]")
print(f"  • Passenger: {len(nodes_passenger):,}개")
print(f"  • PClass: {len(nodes_pclass):,}개 (1st/2nd/3rd)")
print(f"  • Cabin: {len(nodes_cabin):,}개")
print(f"  • Port: {len(nodes_port):,}개 (S/C/Q)")
print(f"  → 총 {len(nodes_passenger) + len(nodes_pclass) + len(nodes_cabin) + len(nodes_port):,}개 노드")

print(f"\n[관계 유형]")
print(f"  • TRAVELED_IN (Passenger → PClass): {len(rels_passenger_pclass):,}개")
print(f"  • STAYED_IN (Passenger → Cabin): {len(rels_passenger_cabin):,}개")
print(f"  • EMBARKED_AT (Passenger → Port): {len(rels_passenger_port):,}개")
if len(rels_traveled_with) > 0:
    print(f"  • TRAVELED_WITH (Passenger ↔ Passenger): {len(rels_traveled_with):,}개")
    total_rels = len(rels_passenger_pclass) + len(rels_passenger_cabin) + len(rels_passenger_port) + len(rels_traveled_with)
else:
    total_rels = len(rels_passenger_pclass) + len(rels_passenger_cabin) + len(rels_passenger_port)
print(f"  → 총 {total_rels:,}개 관계")

print(f"\n저장 위치: {output_dir}/")

타이타닉 지식그래프 구조

[노드 유형]
  • Passenger: 891개
  • PClass: 3개 (1st/2nd/3rd)
  • Cabin: 147개
  • Port: 3개 (S/C/Q)
  → 총 1,044개 노드

[관계 유형]
  • TRAVELED_IN (Passenger → PClass): 891개
  • STAYED_IN (Passenger → Cabin): 204개
  • EMBARKED_AT (Passenger → Port): 889개
  • TRAVELED_WITH (Passenger ↔ Passenger): 351개
  → 총 2,335개 관계

저장 위치: ./output/
